# Praktikum Pengolahan Citra Digital - Sesi 5 & 6
Jalankan sel dari atas ke bawah. Sel pertama untuk upload gambar.

In [ ]:
# SETUP: upload gambar (pilih 1 foto dari HP)
import numpy as np
import imageio.v3 as img
import matplotlib.pyplot as plt

try:
    from google.colab import files
    up = files.upload()
    path = list(up.keys())[0]
except ImportError:
    path = "source.jpg"   # kalau dijalankan di luar Colab

image = img.imread(path)
if image.ndim == 2:                      # kalau gambar grayscale, jadikan 3 channel
    image = np.stack([image]*3, axis=-1)
image = image[:, :, :3]

# kecilkan kalau terlalu besar supaya loop cepat
if max(image.shape[:2]) > 500:
    s = 500 / max(image.shape[:2])
    h, w = image.shape[:2]
    ys = (np.arange(int(h*s)) / s).astype(int)
    xs = (np.arange(int(w*s)) / s).astype(int)
    image = image[ys][:, xs]
print("Ukuran gambar:", image.shape)

## Soal 1.a - Translasi Citra

In [ ]:
def Translasi(image, shiftX, shiftY):
    imgTranslasi = np.roll(image, shift=shiftY, axis=0)          # Geser vertikal
    imgTranslasi = np.roll(imgTranslasi, shift=shiftX, axis=1)   # Geser horizontal

    # Mengisi bagian yang kosong dengan warna hitam (0)
    if shiftY > 0:
        imgTranslasi[:shiftY, :] = 0
    elif shiftY < 0:
        imgTranslasi[shiftY:, :] = 0
    if shiftX > 0:
        imgTranslasi[:, :shiftX] = 0
    elif shiftX < 0:
        imgTranslasi[:, shiftX:] = 0
    return imgTranslasi

imgResult = Translasi(image, shiftX=50, shiftY=-100)

plt.figure(figsize=(10, 10))
plt.subplot(2, 1, 1); plt.imshow(image); plt.title("Asli")
plt.subplot(2, 1, 2); plt.imshow(imgResult); plt.title("Translasi")
plt.show()

## Soal 1.b - Mirroring (horizontal & vertikal terpisah)

In [ ]:
height, width = image.shape[:2]
horizontal = np.zeros_like(image)
vertical = np.zeros_like(image)

for y in range(height):
    for x in range(width):
        horizontal[y, x] = image[y, width - 1 - x]

for y in range(height):
    for x in range(width):
        vertical[y, x] = image[height - 1 - y, x]

plt.figure(figsize=(10, 5))
plt.subplot(1, 3, 1); plt.imshow(image); plt.title("Asli")
plt.subplot(1, 3, 2); plt.imshow(horizontal); plt.title("Horizontal")
plt.subplot(1, 3, 3); plt.imshow(vertical); plt.title("Vertikal")
plt.show()

## Soal 1.c - Rotasi Citra (pivot di tengah)

In [ ]:
def rotateImage(image, degree):
    radian_deg = np.radians(degree)
    cos_deg, sin_deg = np.cos(radian_deg), np.sin(radian_deg)

    height, width = image.shape[:2]
    max_dim = int(np.sqrt(height**2 + width**2))
    outputImage = np.zeros((max_dim, max_dim, 3), dtype=image.dtype)

    centerY, centerX = max_dim // 2, max_dim // 2

    for y in range(-height // 2, height // 2):
        for x in range(-width // 2, width // 2):
            newX = int(cos_deg * x - sin_deg * y) + centerX
            newY = int(sin_deg * x + cos_deg * y) + centerY

            if 0 <= newX < max_dim and 0 <= newY < max_dim:
                outputImage[newY, newX] = image[y + height // 2, x + width // 2]
    return outputImage

rotated_image = rotateImage(image, 45)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(image); plt.title("Asli")
plt.subplot(1, 2, 2); plt.imshow(rotated_image); plt.title("Rotasi 45 derajat")
plt.show()

## Soal 1.d - Zooming (Zoom Plus / memperbesar)

In [ ]:
def zoomPlus(image, factor):
    height, width = image.shape[:2]
    new_height = int(height * factor)
    new_width = int(width * factor)
    imgZoom = np.zeros((new_height, new_width, 3), dtype=image.dtype)

    for y in range(new_height):
        for x in range(new_width):
            ori_y = min(int(y / factor), height - 1)
            ori_x = min(int(x / factor), width - 1)
            imgZoom[y, x] = image[ori_y, ori_x]
    return imgZoom

skala = 2.0
imgZoom = zoomPlus(image, skala)
img.imwrite("zoom_plus.jpg", imgZoom)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(image); plt.title("Asli")
plt.subplot(1, 2, 2); plt.imshow(imgZoom); plt.title(f"Zoom Plus x{skala}")
plt.show()

## Soal 2.a - Mirroring vertikal & horizontal SECARA BERSAMAAN

In [ ]:
height, width = image.shape[:2]
mirrorBoth = np.zeros_like(image)

for y in range(height):
    for x in range(width):
        mirrorBoth[y, x] = image[height - 1 - y, width - 1 - x]

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(image); plt.title("Asli")
plt.subplot(1, 2, 2); plt.imshow(mirrorBoth); plt.title("Mirror Vertikal + Horizontal")
plt.show()

## Soal 2.b - Rotasi dengan pivot di titik (0,0) pojok kiri atas

In [ ]:
def rotateImagePivot00(image, degree):
    radian_deg = np.radians(degree)
    cos_deg, sin_deg = np.cos(radian_deg), np.sin(radian_deg)

    height, width = image.shape[:2]
    max_dim = int(np.sqrt(height**2 + width**2))
    outputImage = np.zeros((max_dim, max_dim, 3), dtype=image.dtype)

    # Pivot (0,0) = pojok kiri atas, jadi x dan y langsung dari 0 (tanpa dikurangi pusat)
    for y in range(height):
        for x in range(width):
            newX = int(cos_deg * x - sin_deg * y)
            newY = int(sin_deg * x + cos_deg * y)

            if 0 <= newX < max_dim and 0 <= newY < max_dim:
                outputImage[newY, newX] = image[y, x]
    return outputImage

# sudut 30 derajat (bagian yang keluar dari batas kiri/atas akan terpotong karena pivot di (0,0))
rotated_pivot = rotateImagePivot00(image, 30)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(image); plt.title("Asli")
plt.subplot(1, 2, 2); plt.imshow(rotated_pivot); plt.title("Rotasi 30 derajat, pivot (0,0)")
plt.show()

## Soal 2.c - Zoom Minus (memperkecil gambar)

In [ ]:
def zoomMinus(image, factor):
    height, width = image.shape[:2]
    new_height = int(height / factor)
    new_width = int(width / factor)
    imgZoom = np.zeros((new_height, new_width, 3), dtype=image.dtype)

    for y in range(new_height):
        for x in range(new_width):
            ori_y = min(int(y * factor), height - 1)
            ori_x = min(int(x * factor), width - 1)
            imgZoom[y, x] = image[ori_y, ori_x]
    return imgZoom

skala = 2.0
imgMinus = zoomMinus(image, skala)
img.imwrite("zoom_minus.jpg", imgMinus)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(image); plt.title("Asli")
plt.subplot(1, 2, 2); plt.imshow(imgMinus); plt.title(f"Zoom Minus /{skala}")
plt.show()